In [6]:
#!/usr/bin/env python
# coding: utf-8

# ============================================
# Block 1: System setup (FMO Hamiltonian, basis)
# ============================================

import numpy as np
from scipy.integrate import quad
from scipy.linalg import expm

# ---------------------------------------------------
# Physical constants and unit conversion (hbar = 1)
# ---------------------------------------------------
C_LIGHT_CM_FS = 2.99792458e-5          # speed of light, cm/fs
CM1_TO_RADFS = 2.0 * np.pi * C_LIGHT_CM_FS   # convert cm^-1 -> rad/fs
KB_CM1_PER_K = 0.695034800             # Boltzmann constant, cm^-1/K

# ---------------------------------------------------
# FMO exciton Hamiltonian (Adolphs & Renger), cm^-1
# ---------------------------------------------------
H_exc_cm1 = np.array([
    [200,  -96,    5,  -4.4,   4.7, -12.6,  -6.2],
    [-96,  320, 33.1,   6.8,   4.5,   7.4,  -0.3],
    [5,   33.1,    0, -51.1,   0.8,  -8.4,   7.6],
    [-4.4,  6.8,-51.1,   110, -76.6, -14.2,   -67],
    [4.7,   4.5,  0.8, -76.6,   270,  78.3,  -0.1],
    [-12.6, 7.4, -8.4, -14.2,  78.3,   420,  38.3],
    [-6.2, -0.3,  7.6,   -67,  -0.1,  38.3,   230]
], dtype=complex)

N_site = H_exc_cm1.shape[0]

# Convert to rad/fs: from now on all energies/rates are in fs^-1
H_exc = H_exc_cm1 * CM1_TO_RADFS

# ---------------------------------------------------
# Diagonalization: site basis -> exciton basis
# ---------------------------------------------------
def diagonalize_system(H_exc):
    """
    Diagonalizes the system Hamiltonian to obtain the exciton basis.

    Returns:
    - eps    : array (N,), exciton energies (eigenvalues), sorted ascending
    - D      : array (N,N), D[:,alpha] = alpha-th eigenvector in the site basis
               (rows = site index, columns = exciton index)
    """
    eps, D = np.linalg.eigh(H_exc)
    return eps, D

eps, D = diagonalize_system(H_exc)

# ---------------------------------------------------
# Site projectors S_i = |i><i| (site basis)
# ---------------------------------------------------
S_site = np.zeros((N_site, N_site, N_site), dtype=complex)
for i in range(N_site):
    S_site[i, i, i] = 1.0

# S_i in the exciton basis: S_i^exc = D^dagger @ S_i @ D
S_exc = np.zeros_like(S_site)
for i in range(N_site):
    S_exc[i] = D.conj().T @ S_site[i] @ D

# s_alpha^(i) = <alpha| S_i |alpha> : diagonal weight, used for the
# Pure Dephasing Kraus operators (real, in [0,1])
s_weights = np.array([np.real(np.diag(S_exc[i])) for i in range(N_site)])  # shape (N_site, N_alpha)

# w_{alpha,beta} = sum_i |<alpha|S_i|beta>|^2  (Eq. V.44 / w_{alpha beta})
def compute_w_alphabeta(S_exc):
    """
    Computes the geometric coupling factor w_{alpha,beta} = sum_i |<alpha|S_i|beta>|^2.
    Returns array of shape (N_alpha, N_beta).
    """
    N = S_exc.shape[1]
    w = np.zeros((N, N))
    for a in range(N):
        for b in range(N):
            w[a, b] = np.sum(np.abs(S_exc[:, a, b])**2)
    return w

w_ab = compute_w_alphabeta(S_exc)


# ============================================
# Block 2: Spectral function C(omega) and Lamb shift
# ============================================

def check_KMS(C_func, omega_test, beta, rtol=1e-8):
    """
    Sanity check of the KMS relation C(-omega) = C(omega) * exp(-beta*omega).
    Returns the relative error (should be ~0).
    """
    lhs = C_func(-omega_test)
    rhs = C_func(omega_test) * np.exp(-beta * omega_test)
    return np.abs(lhs - rhs) / (np.abs(rhs) + 1e-30)


def bose_factor_stable(x):
    """
    Numerically stable computation of 1/(1 - exp(-x)) for any real x
    (avoids overflow for large |x|).
    """
    scalar_input = np.isscalar(x) or np.asarray(x).ndim == 0
    x = np.atleast_1d(np.asarray(x, dtype=float))
    out = np.empty_like(x)

    pos = x > 0
    out[pos] = 1.0 / (1.0 - np.exp(-x[pos]))

    neg = ~pos
    ex = np.exp(x[neg])          # x[neg] <= 0, so ex in (0,1], safe
    out[neg] = ex / (ex - 1.0)

    return out[0] if scalar_input else out


def C_drude_lorentz(omega, lam, Omega, beta, omega_tol=1e-10):
    """
    Drude-Lorentz spectral function C(omega) = 4*lam*omega*Omega/(omega^2+Omega^2) * 1/(1-exp(-omega*beta)).
    Handles the omega -> 0 limit analytically, and uses a numerically stable
    Bose factor to avoid overflow for large |omega|.
    """
    scalar_input = np.isscalar(omega) or np.asarray(omega).ndim == 0
    omega = np.atleast_1d(np.asarray(omega, dtype=float))
    out = np.empty_like(omega)

    small = np.abs(omega) < omega_tol
    out[small] = 4.0 * lam / (Omega * beta)

    w = omega[~small]
    if w.size > 0:
        lorentz = 4.0 * lam * w * Omega / (w**2 + Omega**2)
        out[~small] = lorentz * bose_factor_stable(w * beta)

    return out[0] if scalar_input else out


def lamb_shift_Lambda(omega, C_func, bound=500.0, limit=500):
    """
    Computes Lambda(omega) = (1/2pi) * P.V. Integral[ C(omega')/(omega-omega'), d omega' ]
    via scipy.integrate.quad with a Cauchy principal-value weight.
    """
    integrand = lambda wp: float(np.real(C_func(wp)))

    pv_result, _ = quad(integrand, -bound, bound, weight='cauchy', wvar=omega,
                         limit=limit)   # 'points' removed, incompatible with weight='cauchy'

    return -pv_result / (2.0 * np.pi)


# ---------------------------------------------------
# Quick sanity checks (run this block to verify Block 1+2)
# ---------------------------------------------------
if __name__ == "__main__":
    print("Exciton energies (cm^-1):", eps / CM1_TO_RADFS)
    print("Exciton energies (rad/fs):", eps)

    # Example Drude-Lorentz parameters (TO BE CONFIRMED against your chapter)
    lam_cm1, Omega_cm1, T_K = 35.0, 106.14, 300.0   # <-- placeholder values, adjust!
    lam = lam_cm1 * CM1_TO_RADFS
    Omega = Omega_cm1 * CM1_TO_RADFS
    beta = 1.0 / (KB_CM1_PER_K * T_K * CM1_TO_RADFS)   # fs

    C_func = lambda w: C_drude_lorentz(w, lam, Omega, beta)

    print("\nC(0) =", C_func(0.0), "fs^-1")
    print("KMS relative error at omega=eps[1]-eps[0]:",
          check_KMS(C_func, eps[1] - eps[0], beta))

    Lambda_0 = lamb_shift_Lambda(0.0, C_func)
    print("Lambda(0) =", Lambda_0, "fs^-1")

Exciton energies (cm^-1): [-28.56015219  74.24104724 148.47470057 244.1423858  268.96639038
 374.12760518 468.60802303]
Exciton energies (rad/fs): [-0.00537974  0.01398443  0.02796746  0.04598792  0.0506639   0.0704726
  0.08826942]

C(0) = 0.0518056740106239 fs^-1
KMS relative error at omega=eps[1]-eps[0]: 1.677655944197346e-16
Lambda(0) = -0.006592612659911318 fs^-1
